<a href="https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
!pip install -q duckdb huggingface_hub pyarrow pandas scikit-learn matplotlib

In [17]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

In [18]:
import duckdb
import pandas as pd
import numpy as np
import os
import json

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [19]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

con.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    INSTALL parquet;
    LOAD parquet;
""")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con.sql(f"""
    CREATE OR REPLACE VIEW fact_content_daily_performance AS
    SELECT *
    FROM read_parquet('{march_path}');
""")

print("ML-10 warehouse connection ready.")

ML-10 warehouse connection ready.


## 1. Ranked actions + reason codes

### Ranked Action Queue

The action queue uses the validated Logistic Regression model from ML-08 to rank content by its measured probability of the decline proxy.

The queue is intended for human review, not automatic publishing or content changes. Each recommendation includes a reason code based on the observed performance signals used by the model.

### Reason Codes

- `HIGH_DECLINE_RISK` — high model probability of the decline proxy.
- `LOW_CTR_HIGH_VISIBILITY` — relatively strong visibility with weak click-through performance.
- `WEAK_SEARCH_POSITION` — relatively poor measured search position.
- `LOW_CURRENT_ENGAGEMENT` — low observed clicks relative to impressions.

The model score determines priority, while the reason code gives the reviewer a simple explanation of why the item was surfaced. These reason codes are decision-support signals, not proof that a specific change will improve performance.

In [20]:
# Section 1 — Build ranked action queue

# Rebuild the ML-08 feature + future outcome dataset
ml08_data = con.sql("""
WITH feature_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_feature,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(DISTINCT report_date) AS gsc_measured_days
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),

future_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_clicks
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_90d,
    f.clicks_feature,
    ROUND(f.avg_position, 2) AS avg_position,
    f.gsc_measured_days,

    CASE
        WHEN w.future_clicks = 0 THEN 1
        ELSE 0
    END AS decline_proxy

FROM feature_window f
INNER JOIN future_window w
    ON f.client_hash_id = w.client_hash_id
   AND f.content_hash_id = w.content_hash_id
""").df()


features = [
    "impressions_90d",
    "clicks_feature",
    "avg_position",
    "gsc_measured_days"
]

X = ml08_data[features].copy()
y = ml08_data["decline_proxy"].copy()
groups = ml08_data["client_hash_id"]


# Honest client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]


# Train validated Logistic Regression
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_proba = model.predict_proba(X_test)[:, 1]


# Build queue from held-out content
queue = ml08_data.iloc[test_idx].copy()

queue["model_probability"] = model_proba

# Calculate CTR safely
queue["ctr_percent"] = np.where(
    queue["impressions_90d"] > 0,
    100 * queue["clicks_feature"] / queue["impressions_90d"],
    np.nan
)


# Human-readable reason code
def assign_reason(row):
    if row["model_probability"] >= 0.80:
        return "HIGH_DECLINE_RISK"
    elif (
        row["impressions_90d"] >= 500
        and row["ctr_percent"] < 0.5
    ):
        return "LOW_CTR_HIGH_VISIBILITY"
    elif row["avg_position"] > 10:
        return "WEAK_SEARCH_POSITION"
    else:
        return "LOW_CURRENT_ENGAGEMENT"


queue["reason_code"] = queue.apply(assign_reason, axis=1)

queue["recommended_action"] = np.where(
    queue["reason_code"] == "LOW_CTR_HIGH_VISIBILITY",
    "Review title and search-result snippet",
    np.where(
        queue["reason_code"] == "WEAK_SEARCH_POSITION",
        "Review content relevance and on-page SEO",
        np.where(
            queue["reason_code"] == "LOW_CURRENT_ENGAGEMENT",
            "Review engagement and content alignment",
            "Review content before prioritizing refresh"
        )
    )
)


# Rank highest-risk content first
queue = queue.sort_values(
    "model_probability",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)


# Display top 20
display(
    queue[
        [
            "priority_rank",
            "content_hash_id",
            "model_probability",
            "reason_code",
            "recommended_action",
            "impressions_90d",
            "clicks_feature",
            "ctr_percent",
            "avg_position"
        ]
    ].head(20)
)

print("Ranked queue rows:", len(queue))
print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,priority_rank,content_hash_id,model_probability,reason_code,recommended_action,impressions_90d,clicks_feature,ctr_percent,avg_position
0,1,content_3577b8b0a1828dd1,0.996639,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,126.00
1,2,content_40cb0423d4905b9a,0.993803,HIGH_DECLINE_RISK,Review content before prioritizing refresh,2.0,0.0,0.0,107.00
2,3,content_bbee43811efaf268,0.992726,HIGH_DECLINE_RISK,Review content before prioritizing refresh,2.0,0.0,0.0,98.50
3,4,content_907de8de971de488,0.992204,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,96.00
4,5,content_f8de04ca056310fa,0.992204,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,96.00
5,6,content_08e5527ba7494e89,0.992204,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,96.00
6,7,content_e7e62017bde50b08,0.991755,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,94.00
7,8,content_0c91573a8d938826,0.991670,HIGH_DECLINE_RISK,Review content before prioritizing refresh,3.0,0.0,0.0,93.67
8,9,content_558568c885912bed,0.990826,HIGH_DECLINE_RISK,Review content before prioritizing refresh,3.0,0.0,0.0,93.00
9,10,content_f4be85ce5e5b7e72,0.989399,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,85.00


Ranked queue rows: 10247

Reason-code counts:
reason_code
HIGH_DECLINE_RISK          4821
LOW_CURRENT_ENGAGEMENT     3273
LOW_CTR_HIGH_VISIBILITY    1517
WEAK_SEARCH_POSITION        636
Name: count, dtype: int64


## 2. Intended use and limits

### Intended Use

The playbook is intended for SEO or content teams to prioritize pages for human review. The model score is a ranking signal that helps decide which content should be investigated first.

The recommended actions are decision-support suggestions. A reviewer should inspect the page, search intent, current content quality, business importance, and recent changes before taking action.

### Limits

The model is based on observed March 2026 performance signals and a future zero-click decline proxy. It does not establish that a page will decline in the future or that a recommended action will improve performance.

The validation was performed on a client-grouped split, and the measured top-K precision was strong in that test. However, performance may change across different clients, time periods, search conditions, or datasets.

The queue should therefore be used for prioritization and investigation, not automatic content changes.

### Archetype → Action Mapping

- **High decline risk:** inspect the page first and determine whether a refresh, consolidation, or no change is appropriate.
- **Low CTR with high visibility:** review the title, snippet, and search-result alignment.
- **Weak search position:** review relevance, content coverage, internal linking, and on-page SEO.
- **Low current engagement:** review whether the page matches user intent and whether the content is still useful.

### Decay / Refresh Insight

Content age can be useful as a directional review signal, but this model does not establish that refreshing older content will cause better performance. Any refresh decision should be based on the page's observed signals, relevance, and human review.

In [21]:
# Section 2 — Intended use, limits, and action coverage

print("Intended use:")
print("Human-reviewed prioritization and decision-support for content investigation.")

print("\nModel limitations:")
print("- Uses observed March 2026 performance features.")
print("- Uses a future zero-click outcome as a decline proxy.")
print("- Does not establish causation or guaranteed future performance.")
print("- Recommendations are not intended for automatic execution.")

print("\nArchetype / reason-code coverage:")
print(queue["reason_code"].value_counts())

print("\nQueue coverage:", len(queue), "content items")
print(
    "High-decline-risk share:",
    round(
        100 * (queue["reason_code"] == "HIGH_DECLINE_RISK").mean(),
        2
    ),
    "%"
)

Intended use:
Human-reviewed prioritization and decision-support for content investigation.

Model limitations:
- Uses observed March 2026 performance features.
- Uses a future zero-click outcome as a decline proxy.
- Does not establish causation or guaranteed future performance.
- Recommendations are not intended for automatic execution.

Archetype / reason-code coverage:
reason_code
HIGH_DECLINE_RISK          4821
LOW_CURRENT_ENGAGEMENT     3273
LOW_CTR_HIGH_VISIBILITY    1517
WEAK_SEARCH_POSITION        636
Name: count, dtype: int64

Queue coverage: 10247 content items
High-decline-risk share: 47.05 %


## 3. Human review + the no-go list

### Human Review Rules

Every recommended action must be reviewed by a person before implementation.

The reviewer should check:

1. Whether the page still matches the intended search intent.
2. Whether the content is accurate, useful, and up to date.
3. Whether recent changes or external events explain the observed performance.
4. Whether the page has important business or strategic value.
5. Whether the recommended action is appropriate for the specific page.
6. Whether the proposed change could negatively affect other pages or the wider site.

The model score should determine review priority, not automatically determine the final action.

### No-Go List

The following should **not** be automated:

- Publishing or deleting content.
- Rewriting titles, content, or metadata without human approval.
- Consolidating or redirecting pages automatically.
- Making claims about search ranking or traffic improvement.
- Treating a high model score as proof that a page will decline.
- Applying the same action to every page with the same reason code.
- Making changes where the page has legal, regulatory, brand, or other high-impact implications without specialist review.

The playbook is therefore a human-in-the-loop recommendation system rather than an autonomous content-management system.

In [22]:
# Section 3 — Human review and no-go checks

review_rules = [
    "Check search intent",
    "Check content accuracy and usefulness",
    "Check recent changes or external events",
    "Check business importance",
    "Check action suitability",
    "Check possible effects on other pages"
]

no_go_actions = [
    "Automatic publishing or deletion",
    "Automatic rewriting",
    "Automatic consolidation or redirects",
    "Guaranteed ranking or traffic claims",
    "Automatic action based only on model score",
    "Applying identical actions without page-level review",
    "High-impact changes without specialist review"
]

print("Human review rules:", len(review_rules))
for i, rule in enumerate(review_rules, 1):
    print(f"{i}. {rule}")

print("\nNo-go actions:", len(no_go_actions))
for i, action in enumerate(no_go_actions, 1):
    print(f"{i}. {action}")

print("\nHuman-in-the-loop requirement: PASS")
print("Automatic content execution: NOT ALLOWED")

Human review rules: 6
1. Check search intent
2. Check content accuracy and usefulness
3. Check recent changes or external events
4. Check business importance
5. Check action suitability
6. Check possible effects on other pages

No-go actions: 7
1. Automatic publishing or deletion
2. Automatic rewriting
3. Automatic consolidation or redirects
4. Guaranteed ranking or traffic claims
5. Automatic action based only on model score
6. Applying identical actions without page-level review
7. High-impact changes without specialist review

Human-in-the-loop requirement: PASS
Automatic content execution: NOT ALLOWED


## 4. Monitoring / retrain triggers

### Monitoring Triggers

The playbook should be monitored for changes in the data and in the usefulness of its recommendations.

I would review the playbook if:

- The distribution of model probabilities changes substantially.
- The distribution of key input features changes substantially.
- Precision@20 or Precision@50 decreases on a newly observed validation period.
- The decline-proxy rate changes substantially over time.
- Reviewers frequently reject the recommended actions.
- A new client or content type behaves differently from the clients represented in the validation data.

### Retrain Triggers

Retraining should be considered when monitoring shows sustained changes rather than a single unusual observation.

A retraining review would be triggered if measured top-K precision falls materially on a new validation period, feature distributions show sustained drift, or the relationship between the observed features and the decline proxy changes.

Retraining should use a new time period and an appropriate grouped or time-aware validation split. A new model should not automatically replace the existing model without human review.

These triggers are monitoring guidelines, not production alert thresholds. The current playbook is a research and decision-support workflow.

In [23]:
# Section 4 — Monitoring and retrain trigger checks

# Use the held-out grouped test set created in Section 1
y_reference = ml08_data.iloc[test_idx]["decline_proxy"].copy()


# Calculate current reference Precision@K
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


current_p20 = precision_at_k(
    y_reference,
    model_proba,
    20
)

current_p50 = precision_at_k(
    y_reference,
    model_proba,
    50
)


# Current feature distributions
feature_summary = ml08_data[features].describe().T[
    ["mean", "std", "min", "max"]
]

print("Current reference metrics:")
print(f"Precision@20: {current_p20:.3f}")
print(f"Precision@50: {current_p50:.3f}")

print("\nCurrent decline-proxy rate:")
print(f"{100 * ml08_data['decline_proxy'].mean():.2f}%")

print("\nCurrent feature distribution reference:")
display(feature_summary)

print("\nMonitoring checks:")

monitoring_checks = [
    "Model probability distribution",
    "Key feature distributions",
    "Precision@20 on new validation periods",
    "Precision@50 on new validation periods",
    "Decline-proxy rate over time",
    "Reviewer acceptance/rejection patterns",
    "Performance on new clients or content types"
]

for i, check in enumerate(monitoring_checks, 1):
    print(f"{i}. {check}")

print("\nRetraining principle:")
print(
    "Consider retraining after sustained measured drift or "
    "performance change, followed by fresh grouped/time-aware validation."
)

Current reference metrics:
Precision@20: 1.000
Precision@50: 0.980

Current decline-proxy rate:
64.05%

Current feature distribution reference:


,mean,std,min,max
impressions_90d,900.674765,2763.944947,1.0,161575.0
clicks_feature,2.717581,13.705023,0.0,2395.0
avg_position,15.755752,17.655176,0.0,310.0
gsc_measured_days,11.410909,4.760916,1.0,15.0



Monitoring checks:
1. Model probability distribution
2. Key feature distributions
3. Precision@20 on new validation periods
4. Precision@50 on new validation periods
5. Decline-proxy rate over time
6. Reviewer acceptance/rejection patterns
7. Performance on new clients or content types

Retraining principle:
Consider retraining after sustained measured drift or performance change, followed by fresh grouped/time-aware validation.


## 5. Exports for the paper

### Paper Exports

The ranked action queue is exported so that the recommendations section of the research paper can reuse the same validated output produced by this notebook.

The queue contains the model priority, reason code, recommended action, and the observed performance signals used to support the recommendation.

The queue CSV is regenerated by the notebook and is kept out of Git because it contains row-level data. A compact metrics JSON is also exported as a traceable receipt for the paper's reported validation numbers.

The exports are decision-support artifacts, not production outputs.

In [24]:
# Section 5 — Export queue and metrics for the paper

# Create output directory
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)


# -----------------------------
# 1. Export ranked action queue
# -----------------------------

queue_export = queue[
    [
        "priority_rank",
        "content_hash_id",
        "model_probability",
        "reason_code",
        "recommended_action",
        "impressions_90d",
        "clicks_feature",
        "ctr_percent",
        "avg_position",
        "gsc_measured_days"
    ]
].copy()

queue_path = os.path.join(
    output_dir,
    "ml10_ranked_action_queue.csv"
)

queue_export.to_csv(
    queue_path,
    index=False
)


# -----------------------------
# 2. Export metrics receipt
# -----------------------------

metrics = {
    "model": "Logistic Regression",
    "validation_split": "client-grouped",
    "random_state": 42,
    "dataset_rows": int(len(ml08_data)),
    "test_rows": int(len(queue)),
    "test_clients": int(
        ml08_data.iloc[test_idx]["client_hash_id"].nunique()
    ),
    "precision_at_20": round(float(current_p20), 3),
    "precision_at_50": round(float(current_p50), 3),
    "decline_proxy_rate": round(
        float(ml08_data["decline_proxy"].mean()),
        4
    ),
    "reason_code_counts": {
        str(k): int(v)
        for k, v in queue["reason_code"].value_counts().items()
    },
    "purpose": "research decision-support"
}

metrics_path = os.path.join(
    output_dir,
    "ml10_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)


# -----------------------------
# 3. Verify exports
# -----------------------------

print("Exported files:")

for path in [queue_path, metrics_path]:
    print(
        f"- {path} | "
        f"{os.path.getsize(path):,} bytes"
    )

print("\nQueue rows exported:", len(queue_export))
print("Metrics receipt exported: PASS")

print("\nExport preview:")
display(queue_export.head(5))

Exported files:
- work/outputs/ml10_ranked_action_queue.csv | 1,419,823 bytes
- work/outputs/ml10_metrics.json | 461 bytes

Queue rows exported: 10247
Metrics receipt exported: PASS

Export preview:


,priority_rank,content_hash_id,model_probability,reason_code,recommended_action,impressions_90d,clicks_feature,ctr_percent,avg_position,gsc_measured_days
0,1,content_3577b8b0a1828dd1,0.996639,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,126.0,1
1,2,content_40cb0423d4905b9a,0.993803,HIGH_DECLINE_RISK,Review content before prioritizing refresh,2.0,0.0,0.0,107.0,2
2,3,content_bbee43811efaf268,0.992726,HIGH_DECLINE_RISK,Review content before prioritizing refresh,2.0,0.0,0.0,98.5,1
3,4,content_907de8de971de488,0.992204,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,96.0,1
4,5,content_f8de04ca056310fa,0.992204,HIGH_DECLINE_RISK,Review content before prioritizing refresh,1.0,0.0,0.0,96.0,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.